# Configuration env

In [1]:
import pandas as pd

In [2]:
from pathlib import Path
import os

# remonte jusqu'au dossier qui contient .git, puis s'y place
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").is_dir())
os.chdir(ROOT)
print("racine projet :", ROOT)

racine projet : /Users/benjaminscemama/dev/market-risk-control


In [3]:
%load_ext sql
%config SqlMagic.autopandas = True
%sql duckdb:///:memory:
%sql ATTACH IF NOT EXISTS 'data/risk.db' AS r (TYPE sqlite);

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

,Success


In [4]:
%%sql
-- Test configuration environnement
SELECT name FROM (SHOW ALL TABLES) WHERE database = 'r' ORDER BY name;

Running query in 'duckdb:///:memory:'

,name
0,mkt_forward_curve
1,mkt_spot_hourly
2,pos_snapshot
3,ref_contract
4,ref_customer
5,ref_site
6,trd_deal


# 1. `ref_customer`

## Niveau 0 - cadrage

In [163]:
%%sql 
-- Nombre de lignes
select count(*) as nb_raw from r.ref_customer;

Running query in 'duckdb:///:memory:'

,nb_raw
0,220


---

In [164]:
%%sql
-- unicité de customer_id
select count(*) as n_raw, count(distinct customer_id) as n_distinct, count(customer_id) as n_customer_id from r.ref_customer;

Running query in 'duckdb:///:memory:'

,n_raw,n_distinct,n_customer_id
0,220,220,220


---

In [165]:
%%sql 
-- Unicité customer_name
select customer_name, count(*) as nb_customer_name from r.ref_customer
group by customer_name
having count(*) > 1;

Running query in 'duckdb:///:memory:'

,customer_name,nb_customer_name


In [166]:
%%sql 
select upper(trim(customer_name)) as customer_name from r.ref_customer
group by upper(trim(customer_name))
having count(*) > 1;

Running query in 'duckdb:///:memory:'

,customer_name


In [167]:
%%sql
with n as ( 
    select 
        customer_name as n0,
        upper(trim(customer_name)) as n1,
        replace(upper(trim(customer_name)), '.', '') as n2,
        regexp_replace(upper(trim(customer_name)), '\s+', ' ', 'g') as n3,
        regexp_replace(strip_accents(upper(trim(customer_name))), '[^A-Z0-9]', '', 'g') as n4
    from r.ref_customer
)
select 
    count(*) as lignes,
    count(distinct n0) as brut,
    count(distinct n1) as upper_trim,
    count(distinct n2) as sans_point,
    count(distinct n3) as espaces_normalises,
    count(distinct n4) as alphanumerique_seul
from n;


Running query in 'duckdb:///:memory:'

,lignes,brut,upper_trim,sans_point,espaces_normalises,alphanumerique_seul
0,220,220,220,220,220,220


---

In [168]:
%%sql
-- Exhaustivité « tout customer_id référencé dans ref_site ou ref_contract figure ici »
select rs.customer_id from r.ref_site as rs
left join r.ref_customer as rc on rs.customer_id = rc.customer_id
where rc.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id


In [169]:
%%sql
-- Exhaustivité « tout customer_id référencé dans ref_site ou ref_contract figure ici »
select rco.customer_id from r.ref_contract as rco
left join r.ref_customer as rc on rco.customer_id = rc.customer_id
where rc.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id


---

In [170]:
%%sql 
-- Tous les clients ont un site
select rc.customer_id from r.ref_customer as rc
left join r.ref_site as rs on rc.customer_id = rs.customer_id
where rs.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id


In [171]:
%%sql
-- Tous les clients ont un contrat ?
select rc.customer_id from r.ref_customer as rc
left join r.ref_contract as rco on rc.customer_id = rco.customer_id
where rco.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id
0,C100035
1,C100132
2,C100046
3,C100103
4,C100116
...,...
69,C100152
70,C100194
71,C100013
72,C100076


---

# Niveau 1 - colonnes

In [172]:
df = %sql select * from r.ref_customer
df.nunique()

Running query in 'duckdb:///:memory:'

customer_id      220
customer_name    220
sector             7
segment            4
credit_rating      6
dtype: int64

In [30]:
df.isna().mean().sort_values(ascending=False)

customer_id      0.0
customer_name    0.0
sector           0.0
segment          0.0
credit_rating    0.0
dtype: float64

In [59]:
for c in ["sector", "segment", "credit_rating"] : 
    print(df[c].value_counts(dropna = False))

sector
Sante              39
Tertiaire          35
Distribution       33
Industrie          33
Collectivite       27
Transport          27
Agroalimentaire    26
Name: count, dtype: int64
segment
PME             74
ETI             66
PUBLIC          43
GRAND_COMPTE    37
Name: count, dtype: int64
credit_rating
BBB    64
BB     54
A      33
NR     31
B      27
AA     11
Name: count, dtype: int64


In [21]:
pd.crosstab(df.segment, df.sector)

sector,Agroalimentaire,Collectivite,Distribution,Industrie,Sante,Tertiaire,Transport
segment,,,,,,,
ETI,10,5,11,8,11,13,8
GRAND_COMPTE,4,3,6,8,6,5,5
PME,8,13,13,8,11,8,13
PUBLIC,4,6,3,9,11,9,1


# 2. `ref_site`

## Niveau 0 - cadrage

In [10]:
%%sql
select 
    count(*) as lignes,
    count(distinct site_id) as sites_distincts,
    count(distinct (site_id, commodity)) as site_commodity,
    count (distinct (site_id, commodity, dso)) as site_commodity_dso,
    count(distinct customer_id) as client
from r.ref_site;

Running query in 'duckdb:///:memory:'

,lignes,sites_distincts,site_commodity,site_commodity_dso,client
0,1400,1400,1400,1400,220


In [17]:
%%sql
select monitored, count(*) as n
from r.ref_site
group by monitored
order by monitored;

Running query in 'duckdb:///:memory:'

,monitored,n
0,0,900
1,1,500


In [19]:
df_ref_site = %sql select * from r.ref_site;
pd.crosstab(df_ref_site.dso, df_ref_site.region)

Running query in 'duckdb:///:memory:'

region,ARA,BRE,CVL,GES,HDF,IDF,NAQ,OCC,PACA,PDL
dso,,,,,,,,,,
ENEDIS,29,28,27,27,30,31,26,25,30,15
GEREDIS,37,37,26,36,33,23,28,24,24,29
GRDF,29,24,26,26,27,34,38,34,26,23
RESEAU_LOCAL,28,22,28,26,22,31,22,30,25,25
SRD,22,31,27,27,24,24,37,31,31,35


In [17]:
pd.crosstab(df_ref_site.dso, df_ref_site.commodity)

commodity,GAS,POWER
dso,,
ENEDIS,102,166
GEREDIS,109,188
GRDF,122,165
RESEAU_LOCAL,93,166
SRD,95,194


In [61]:
ko = df_ref_site.query("(dso == 'ENEDIS' and commodity == 'GAS') or (dso == 'GRDF' and commodity == 'POWER')")

print("lignes fautives :", len(ko), "sur", len(df_ref_site))
print("part des lignes :", len(ko) / len(df_ref_site))
print("puissance fautive :", ko.contracted_capacity_kw.sum(), "kW sur", df_ref_site.contracted_capacity_kw.sum())
print("part de la puissance :", ko.contracted_capacity_kw.sum() / df_ref_site.contracted_capacity_kw.sum())

lignes fautives : 267 sur 1400
part des lignes : 0.19071428571428573
puissance fautive : 1125523.0 kW sur 5242077.0
part de la puissance : 0.21470936043098948


In [77]:
print(f'Nombre de lignes nulles capacité kW : {df_ref_site["contracted_capacity_kw"].isna().sum()}')

Nombre de lignes nulles capacité kW : 0


In [79]:
print(f'Nombre de lignes kW neg : {df_ref_site["contracted_capacity_kw"].le(0).sum()}')

Nombre de lignes kW neg : 0


In [47]:
df_ref_site["contracted_capacity_kw"].describe()

count     1400.000000
mean      3744.340714
std       5057.851887
min         61.000000
25%       1157.500000
50%       2258.000000
75%       4538.750000
max      89750.000000
Name: contracted_capacity_kw, dtype: float64

In [88]:
g = df_ref_site.groupby("monitored")["contracted_capacity_kw"].agg(["count", "median", "mean", "sum"])
g["part_puissance"] = g["sum"] / g["sum"].sum()
g

,count,median,mean,sum,part_puissance
monitored,,,,,
0,900,2190.5,3693.991111,3324592.0,0.634213
1,500,2338.5,3834.970000,1917485.0,0.365787


In [97]:
pd.crosstab(df_ref_site.monitored, df_ref_site.commodity)

commodity,GAS,POWER
monitored,,
0,337,563
1,184,316


In [98]:
pd.crosstab(df_ref_site.monitored, df_ref_site.profile_type)

profile_type,PROFILE_BASE,PROFILE_FLAT,PROFILE_PEAK,PROFILE_SEASONAL
monitored,,,,
0,220,234,231,215
1,126,130,122,122


In [114]:
gros = df_ref_site.query("contracted_capacity_kw > 50000")
len(gros), gros["contracted_capacity_kw"].sum() / df_ref_site["contracted_capacity_kw"].sum()

(2, np.float64(0.027658502536303836))

In [123]:
df_ref_site["contracted_capacity_kw"].nlargest(20)

318     89750.0
438     55238.0
313     38433.0
560     33640.0
785     29264.0
651     28482.0
743     28341.0
241     27581.0
1153    27288.0
335     26813.0
761     26338.0
551     25072.0
712     24839.0
538     24561.0
252     24531.0
1344    24496.0
568     24219.0
76      24141.0
1006    23654.0
1003    22976.0
Name: contracted_capacity_kw, dtype: float64

In [126]:
pd.crosstab(df_ref_site.profile_type, df_ref_site.commodity, normalize="index")

commodity,GAS,POWER
profile_type,,
PROFILE_BASE,0.372832,0.627168
PROFILE_FLAT,0.381868,0.618132
PROFILE_PEAK,0.371105,0.628895
PROFILE_SEASONAL,0.362018,0.637982


In [141]:
pd.crosstab(df_ref_site.monitored, df_ref_site.profile_type, normalize = "index")

profile_type,PROFILE_BASE,PROFILE_FLAT,PROFILE_PEAK,PROFILE_SEASONAL
monitored,,,,
0,0.244444,0.26,0.256667,0.238889
1,0.252000,0.26,0.244000,0.244000


In [162]:
df_ref_site

,site_id,customer_id,commodity,region,dso,contracted_capacity_kw,profile_type,monitored
0,S500000,C100192,POWER,CVL,SRD,651.0,PROFILE_BASE,0
1,S500001,C100026,GAS,BRE,GEREDIS,5576.0,PROFILE_FLAT,1
2,S500002,C100122,POWER,NAQ,GEREDIS,1822.0,PROFILE_BASE,1
3,S500003,C100090,POWER,GES,RESEAU_LOCAL,694.0,PROFILE_BASE,1
4,S500004,C100198,POWER,CVL,GEREDIS,1044.0,PROFILE_FLAT,1
...,...,...,...,...,...,...,...,...
1395,S501395,C100193,POWER,ARA,GEREDIS,2047.0,PROFILE_FLAT,1
1396,S501396,C100178,POWER,PDL,SRD,12895.0,PROFILE_SEASONAL,0
1397,S501397,C100112,POWER,CVL,RESEAU_LOCAL,437.0,PROFILE_PEAK,1
1398,S501398,C100138,POWER,CVL,RESEAU_LOCAL,4227.0,PROFILE_SEASONAL,1


In [221]:
%%sql
-- customers qui ont plusieurs sites par commodité
select customer_id, commodity, count(*) as nb from r.ref_site 
group by commodity, customer_id
having count(*) > 1
order by count(*), commodity asc

Running query in 'duckdb:///:memory:'

,customer_id,commodity,nb
0,C100037,GAS,2
1,C100212,GAS,2
2,C100004,GAS,2
3,C100069,GAS,2
4,C100157,GAS,2
...,...,...,...
336,C100182,POWER,8
337,C100027,POWER,8
338,C100127,POWER,9
339,C100178,POWER,9


In [104]:
%%sql
-- Nombre de customer qui ont plusieurs site (bi-energie)
select count(*) as nb_clients_bi_energie 
from (
    select customer_id from r.ref_site 
    where commodity in ('GAS', 'POWER')
    group by customer_id
    having count(distinct commodity) = 2
);

Running query in 'duckdb:///:memory:'

,nb_clients_bi_energie
0,188


In [7]:
%%sql
select count(*) as nb_clients_bi_energie
from (
    select customer_id
    from r.ref_contract
    where commodity in ('GAS', 'POWER')
    group by customer_id
    having count(distinct commodity) = 2
) t;

Running query in 'duckdb:///:memory:'

,nb_clients_bi_energie
0,37


# 3. `ref_contract`

In [65]:
%%sql 
select 
    count(*) as lignes,
    count(distinct customer_id) as clients,
    count(distinct contract_id) as contrats_distintcs, 
    count(*) filter (where end_date < '2026-07-24') as expires,
    count(*) filter (where start_date > '2026-07-24') as a_venir,
    count(*) filter (where start_date <= '2026-07-24' and end_date >= '2026-07-24') as en_vigueur
from r.ref_contract;


Running query in 'duckdb:///:memory:'

,lignes,clients,contrats_distintcs,expires,a_venir,en_vigueur
0,260,146,260,73,0,187


In [43]:
%%sql
select n_contrats, count(*) as n_clients
from (
    select customer_id, count(*) as n_contrats
    from r.ref_contract
    group by customer_id
)
group by n_contrats
order by n_contrats;

Running query in 'duckdb:///:memory:'

,n_contrats,n_clients
0,1,76
1,2,39
2,3,22
3,4,7
4,5,1
5,7,1


In [45]:
%%sql
select count(*) as couples_client_commodite
from (select distinct customer_id, commodity from r.ref_contract);

Running query in 'duckdb:///:memory:'

,couples_client_commodite
0,183


In [46]:
%%sql
select customer_id, commodity, count(*) as n_contrats_actifs
from r.ref_contract
where start_date <= '2026-07-24' and end_date >= '2026-07-24'
group by customer_id, commodity
having count(*) > 1
order by n_contrats_actifs desc;

Running query in 'duckdb:///:memory:'

,customer_id,commodity,n_contrats_actifs
0,C100190,POWER,3
1,C100209,POWER,3
2,C100107,POWER,3
3,C100179,POWER,3
4,C100100,POWER,3
5,C100006,POWER,3
6,C100198,POWER,2
7,C100140,POWER,2
8,C100124,POWER,2
9,C100066,POWER,2


In [47]:
%%sql
with couples as (
    select distinct customer_id, commodity from r.ref_contract
),
actifs as (
    select customer_id, commodity, count(*) as n
    from r.ref_contract
    where start_date <= '2026-07-24' and end_date >= '2026-07-24'
    group by customer_id, commodity
)
select
    (select count(*) from couples)                 as couples_total,
    (select count(*) from actifs)                  as couples_avec_contrat_actif,
    (select count(*) from couples)
        - (select count(*) from actifs)            as couples_sans_contrat_actif,
    (select count(*) from actifs where n > 1)      as couples_en_chevauchement,
    (select sum(n - 1) from actifs where n > 1)    as contrats_en_trop;

Running query in 'duckdb:///:memory:'

,couples_total,couples_avec_contrat_actif,couples_sans_contrat_actif,couples_en_chevauchement,contrats_en_trop
0,183,146,37,35,41.0


In [63]:
%%sql
select * from r.ref_contract;

Running query in 'duckdb:///:memory:'

,contract_id,customer_id,commodity,start_date,end_date,pricing_type,volume_tolerance_pct
0,K7000,C100210,GAS,2025-12-01,2027-12-01,FIXED,20
1,K7001,C100075,POWER,2025-04-01,2026-04-01,FIXED,10
2,K7002,C100097,POWER,2024-01-01,2027-01-01,INDEXED,10
3,K7003,C100171,POWER,2025-07-01,2026-07-01,INDEXED,15
4,K7004,C100029,GAS,2024-12-01,2026-12-01,FIXED,15
...,...,...,...,...,...,...,...
255,K7255,C100105,POWER,2025-12-01,2028-12-01,INDEXED,10
256,K7256,C100038,POWER,2024-09-01,2026-09-01,CLICK,20
257,K7257,C100145,POWER,2024-07-01,2027-07-01,INDEXED,20
258,K7258,C100086,POWER,2026-01-01,2029-01-01,FIXED,15


In [67]:
%%sql 
select 
    count(*) as lignes,
    count(*) filter (where start_date > end_date) as dates_inversees,
    min(date_diff('year', start_date::date, end_date::date)) as duree_min_an,
    median(date_diff('year', start_date::date, end_date::date)) as duree_med_an,
    max(date_diff('year', start_date::date, end_date::date)) as duree_max_jours,
from r.ref_contract;

Running query in 'duckdb:///:memory:'

,lignes,dates_inversees,duree_min_an,duree_med_an,duree_max_jours
0,260,0,1,2.0,3


In [69]:
%%sql 
select 
    count(*) as lignes, 
    count(volume_tolerance_pct) as tol_non_nulles,
    count(*) filter (where volume_tolerance_pct <= 0) as tol_negatives_ou_nulles,
    min(volume_tolerance_pct) as tol_min, 
    max(volume_tolerance_pct) as tol_max, 
    count(distinct commodity) as n_commodity,
    count(distinct pricing_type) as n_pricing_type
from r.ref_contract;

Running query in 'duckdb:///:memory:'

,lignes,tol_non_nulles,tol_negatives_ou_nulles,tol_min,tol_max,n_commodity,n_pricing_type
0,260,260,0,5,20,2,4


In [70]:
%%sql
select distinct pricing_type from r.ref_contract;

Running query in 'duckdb:///:memory:'

,pricing_type
0,INDEXED
1,CLICK
2,FIXED
3,SPOT_PASSTHROUGH


In [74]:
%%sql 
select commodity, pricing_type, count(*) as n
from r.ref_contract
group by all
order by n desc;

Running query in 'duckdb:///:memory:'

,commodity,pricing_type,n
0,POWER,FIXED,75
1,GAS,FIXED,40
2,POWER,INDEXED,36
3,POWER,CLICK,34
4,GAS,INDEXED,23
5,POWER,SPOT_PASSTHROUGH,22
6,GAS,CLICK,20
7,GAS,SPOT_PASSTHROUGH,10


In [76]:
%%sql
select volume_tolerance_pct, count(*) as n
from r.ref_contract
group by all
order by n desc;

Running query in 'duckdb:///:memory:'

,volume_tolerance_pct,n
0,10,78
1,15,65
2,20,64
3,5,53


In [126]:
df = %sql select * from r.ref_contract
pd.crosstab(df.customer_id, df.commodity).value_counts()

Running query in 'duckdb:///:memory:'

GAS  POWER
0    1        49
1    0        27
0    2        20
1    1        16
2    1         8
0    3         6
1    2         6
     3         3
2    0         3
3    0         2
0    4         1
2    2         1
     3         1
     5         1
3    1         1
4    0         1
Name: count, dtype: int64

In [135]:
sites    = %sql select * from r.ref_site
contrats = %sql select * from r.ref_contract

REF = "2026-07-24"
actifs = contrats.query("start_date <= @REF and end_date >= @REF")

Running query in 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

In [139]:
c_site = sites[["customer_id", "commodity"]].drop_duplicates()
c_ctr  = actifs[["customer_id", "commodity"]].drop_duplicates()

c_site.merge(c_ctr, on=["customer_id", "commodity"], how="outer", indicator=True)["_merge"].value_counts()

_merge
left_only     270
both          138
right_only      8
Name: count, dtype: int64

In [137]:
j = sites.merge(actifs, on=["customer_id", "commodity"], how="left", indicator=True)

orph = j[j["_merge"] == "left_only"]

print("lignes jointure externe :", len(j))
print("lignes jointure interne :", (j["_merge"] == "both").sum())
print("sites orphelins         :", orph["site_id"].nunique())
print("kW orphelins            :", orph["contracted_capacity_kw"].sum())
print("part kW                 :", orph["contracted_capacity_kw"].sum() / sites["contracted_capacity_kw"].sum())

lignes jointure externe : 1547
lignes jointure interne : 648
sites orphelins         : 899
kW orphelins            : 3402619.0
part kW                 : 0.6490974855958812
